In [ ]:

# ================================================================
# STREAMING EVENT PROCESSING PIPELINE
# Google Colab | Python | Pandas | NumPy
# ================================================================

import json
import time
import uuid
import shutil
import logging
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# -------------------- 1. PROJECT CONFIGURATION --------------------

PROJECT_DIR = Path("/content/streaming_event_pipeline")

def configure_project(root):
    global PROJECT_DIR, DATA_DIR, OUTPUT_DIR, STATE_DIR, LOG_DIR
    global QUARANTINE_DIR, SOURCE_FILE, CHECKPOINT_FILE
    global ACCEPTED_FILE, REJECTED_FILE, AGGREGATE_FILE, RUN_LOG_FILE

    PROJECT_DIR = Path(root)
    DATA_DIR = PROJECT_DIR / "data"
    OUTPUT_DIR = PROJECT_DIR / "output"
    STATE_DIR = PROJECT_DIR / "state"
    LOG_DIR = PROJECT_DIR / "logs"
    QUARANTINE_DIR = OUTPUT_DIR / "quarantine"

    SOURCE_FILE = DATA_DIR / "events.csv"
    CHECKPOINT_FILE = STATE_DIR / "checkpoint.json"
    ACCEPTED_FILE = OUTPUT_DIR / "accepted_events.csv"
    REJECTED_FILE = QUARANTINE_DIR / "rejected_events.csv"
    AGGREGATE_FILE = OUTPUT_DIR / "aggregates.csv"
    RUN_LOG_FILE = LOG_DIR / "run_metrics.csv"

    for folder in [
        DATA_DIR, OUTPUT_DIR, STATE_DIR, LOG_DIR, QUARANTINE_DIR
    ]:
        folder.mkdir(parents=True, exist_ok=True)

configure_project(PROJECT_DIR)

BATCH_SIZE = 500
ALLOWED_LATENESS_MINUTES = 10
RANDOM_SEED = 42

REQUIRED_COLUMNS = [
    "event_id", "order_id", "customer_id",
    "event_time", "amount", "status", "source_offset"
]

VALID_STATUSES = {"created", "paid", "cancelled"}
VALID_CUSTOMERS = {f"CUST{i:05d}" for i in range(1, 1001)}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)
logger = logging.getLogger("streaming_pipeline")

def utc_now():
    return datetime.now(timezone.utc).isoformat()


# -------------------- 2. SYNTHETIC EVENT PRODUCER --------------------

def generate_events(n=1000, seed=RANDOM_SEED, include_anomalies=True):
    """Generate reproducible synthetic order events."""

    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2026-10-01T00:00:00Z")

    df = pd.DataFrame({
        "event_id": [f"EVT{i:08d}" for i in range(1, n + 1)],
        "order_id": [f"ORD{i:08d}" for i in range(1, n + 1)],
        "customer_id": [
            f"CUST{x:05d}" for x in rng.integers(1, 1001, size=n)
        ],
        "event_time": [
            (start + pd.Timedelta(seconds=int(x))).isoformat()
            for x in rng.integers(0, 7 * 24 * 3600, size=n)
        ],
        "amount": np.round(rng.uniform(10, 5000, size=n), 2),
        "status": rng.choice(
            ["created", "paid", "cancelled"],
            size=n,
            p=[0.25, 0.60, 0.15]
        ),
        "source_offset": np.arange(1, n + 1, dtype=int)
    }, columns=REQUIRED_COLUMNS)

    if include_anomalies and n >= 20:
        # Duplicate event delivery, but a different source offset
        duplicate = df.iloc[[2]].copy()
        duplicate["source_offset"] = n + 1
        df = pd.concat([df, duplicate], ignore_index=True)

        # Missing customer
        df.loc[5, "customer_id"] = None

        # Unknown customer reference
        df.loc[7, "customer_id"] = "CUST99999"

        # Negative amount
        df.loc[9, "amount"] = -25.0

        # Invalid status
        df.loc[11, "status"] = "unknown"

        # Late event
        df.loc[13, "event_time"] = "2026-09-01T00:00:00Z"

    return df


# -------------------- 3. SCHEMA AND VALIDATION --------------------

def validate_schema(df):
    missing = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
    unexpected = sorted(set(df.columns) - set(REQUIRED_COLUMNS))

    if missing or unexpected:
        raise ValueError(
            f"Schema mismatch: missing={missing}, unexpected={unexpected}"
        )

    # Validate the source offset column before processing.
    offsets = pd.to_numeric(df["source_offset"], errors="coerce")

    if offsets.isna().any():
        raise ValueError("Malformed source_offset values")

    if not np.isfinite(offsets.to_numpy(dtype=float)).all():
        raise ValueError("Non-finite source_offset values")

    if ((offsets % 1) != 0).any() or (offsets < 1).any():
        raise ValueError("source_offset must be a positive integer")

    if offsets.duplicated().any():
        raise ValueError("Duplicate source_offset in source")

    df["source_offset"] = offsets.astype("int64")


def validate_record(row):
    errors = []

    for col in [
        "event_id", "order_id", "customer_id", "event_time", "status"
    ]:
        value = row[col]
        if pd.isna(value) or not str(value).strip():
            errors.append(f"missing_{col}")

    if not pd.isna(row["event_id"]):
        if not str(row["event_id"]).strip():
            errors.append("missing_event_id")

    if not pd.isna(row["customer_id"]):
        if str(row["customer_id"]).strip() not in VALID_CUSTOMERS:
            errors.append("invalid_customer_reference")

    if not pd.isna(row["status"]):
        if str(row["status"]).strip() not in VALID_STATUSES:
            errors.append("invalid_status")

    try:
        amount = float(row["amount"])
        if not np.isfinite(amount):
            errors.append("amount_not_finite")
        elif amount < 0:
            errors.append("negative_amount")
    except (ValueError, TypeError, OverflowError):
        errors.append("invalid_amount")

    try:
        event_time = pd.to_datetime(row["event_time"], utc=True, errors="raise")
        if pd.isna(event_time):
            errors.append("invalid_event_time")
    except (ValueError, TypeError, OverflowError):
        errors.append("invalid_event_time")

    try:
        offset = float(row["source_offset"])
        if not np.isfinite(offset) or offset < 1 or not offset.is_integer():
            errors.append("invalid_source_offset")
    except (ValueError, TypeError, OverflowError):
        errors.append("invalid_source_offset")

    return sorted(set(errors))


# -------------------- 4. CHECKPOINT AND FILE HELPERS --------------------

def read_checkpoint():
    if not CHECKPOINT_FILE.exists():
        return {
            "last_source_offset": 0,
            "max_event_time_seen": None
        }

    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def write_checkpoint(state):
    temp_path = CHECKPOINT_FILE.with_suffix(".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)

    temp_path.replace(CHECKPOINT_FILE)


def read_existing(path):
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)
    return pd.DataFrame()


def write_csv_atomic(df, path):
    """Replace a CSV atomically within the local filesystem."""
    temp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(temp_path, index=False)
    temp_path.replace(path)


def append_unique_rows(new_df, path, key):
    """Persist records without repeating a previously written key."""
    if new_df.empty:
        return

    old_df = read_existing(path)

    if not old_df.empty:
        known = set(old_df[key].astype(str))
        new_df = new_df[~new_df[key].astype(str).isin(known)]

    if new_df.empty:
        return

    combined = pd.concat([old_df, new_df], ignore_index=True)
    combined = combined.drop_duplicates(subset=[key], keep="first")
    write_csv_atomic(combined, path)


# -------------------- 5. AGGREGATE PUBLISHING --------------------

def publish_aggregates():
    accepted = read_existing(ACCEPTED_FILE)

    if accepted.empty:
        columns = [
            "event_hour", "status", "event_count", "total_amount"
        ]
        write_csv_atomic(pd.DataFrame(columns=columns), AGGREGATE_FILE)
        return

    accepted["event_time"] = pd.to_datetime(
        accepted["event_time"], utc=True, errors="raise"
    )
    accepted["amount"] = pd.to_numeric(accepted["amount"], errors="raise")
    accepted["event_hour"] = (
        accepted["event_time"].dt.floor("h").astype(str)
    )

    aggregates = (
        accepted.groupby(["event_hour", "status"], as_index=False)
        .agg(
            event_count=("event_id", "count"),
            total_amount=("amount", "sum")
        )
    )

    aggregates["total_amount"] = aggregates["total_amount"].round(2)
    write_csv_atomic(aggregates, AGGREGATE_FILE)


# -------------------- 6. MICRO-BATCH PIPELINE --------------------

def process_pipeline(batch_size=BATCH_SIZE, fail_after_write=False):
    run_id = str(uuid.uuid4())
    started_at = utc_now()
    start_clock = time.perf_counter()

    input_count = 0
    accepted_count = 0
    rejected_count = 0
    duplicate_count = 0
    status = "success"
    error_message = None

    state = read_checkpoint()
    last_offset = int(state["last_source_offset"])
    previous_max = state.get("max_event_time_seen")

    try:
        source = pd.read_csv(SOURCE_FILE, keep_default_na=True)
        validate_schema(source)
        source = source.sort_values("source_offset").reset_index(drop=True)

        pending = source[source["source_offset"] > last_offset].copy()

        for batch_start in range(0, len(pending), batch_size):
            batch = pending.iloc[batch_start:batch_start + batch_size]

            if batch.empty:
                continue

            input_count += len(batch)

            accepted_old = read_existing(ACCEPTED_FILE)
            rejected_old = read_existing(REJECTED_FILE)

            # Recover safely if output writes succeeded but the checkpoint
            # was not updated before a crash.
            processed_offsets = set()

            for old_df in [accepted_old, rejected_old]:
                if not old_df.empty:
                    processed_offsets.update(
                        pd.to_numeric(
                            old_df["source_offset"], errors="coerce"
                        ).dropna().astype(int).tolist()
                    )

            seen_event_ids = (
                set(accepted_old["event_id"].astype(str))
                if not accepted_old.empty else set()
            )

            batch_seen = set()
            accepted_rows = []
            rejected_rows = []

            try:
                running_max = (
                    pd.Timestamp(previous_max)
                    if previous_max else None
                )
            except (ValueError, TypeError):
                running_max = None

            # Process in source order; watermark advances as event times arrive.
            for _, row in batch.iterrows():
                offset = int(row["source_offset"])

                if offset in processed_offsets:
                    continue

                reasons = validate_record(row)

                event_id = (
                    str(row["event_id"]).strip()
                    if not pd.isna(row["event_id"]) else ""
                )

                if not reasons:
                    if event_id in seen_event_ids or event_id in batch_seen:
                        reasons.append("duplicate_event_id")
                        duplicate_count += 1
                    else:
                        batch_seen.add(event_id)

                parsed_time = pd.to_datetime(
                    row["event_time"], utc=True, errors="coerce"
                )

                if reasons:
                    rejected_rows.append({
                        **row.to_dict(),
                        "run_id": run_id,
                        "reject_reason": "|".join(sorted(set(reasons)))
                    })
                    continue

                is_late = (
                    running_max is not None
                    and parsed_time <
                    running_max - pd.Timedelta(
                        minutes=ALLOWED_LATENESS_MINUTES
                    )
                )

                accepted_rows.append({
                    **row.to_dict(),
                    "event_time": parsed_time.isoformat(),
                    "amount": float(row["amount"]),
                    "source_offset": offset,
                    "run_id": run_id,
                    "is_late": bool(is_late)
                })

                if running_max is None or parsed_time > running_max:
                    running_max = parsed_time

            accepted_new = pd.DataFrame(accepted_rows)
            rejected_new = pd.DataFrame(rejected_rows)

            # Write output before advancing the checkpoint.
            append_unique_rows(
                accepted_new, ACCEPTED_FILE, "source_offset"
            )
            append_unique_rows(
                rejected_new, REJECTED_FILE, "source_offset"
            )

            accepted_count += len(accepted_new)
            rejected_count += len(rejected_new)

            # Used to demonstrate retry behaviour in the tests.
            if fail_after_write:
                fail_after_write = False
                raise RuntimeError("Simulated failure before checkpoint")

            last_offset = int(batch["source_offset"].max())

            if running_max is not None:
                previous_max = running_max.isoformat()

            write_checkpoint({
                "last_source_offset": last_offset,
                "max_event_time_seen": previous_max
            })

        publish_aggregates()

    except Exception as exc:
        status = "failed"
        error_message = str(exc)
        logger.exception("Pipeline run failed: %s", run_id)

    elapsed = round(time.perf_counter() - start_clock, 4)

    metrics = {
        "run_id": run_id,
        "started_at": started_at,
        "ended_at": utc_now(),
        "status": status,
        "input_count": input_count,
        "accepted_count": accepted_count,
        "rejected_count": rejected_count,
        "duplicate_count": duplicate_count,
        "last_source_offset": last_offset,
        "elapsed_seconds": elapsed,
        "error_message": error_message
    }

    append_unique_rows(
        pd.DataFrame([metrics]), RUN_LOG_FILE, "run_id"
    )

    print(json.dumps(metrics, indent=2, default=str))
    return metrics


# -------------------- 7. RESET HELPERS --------------------

def reset_outputs():
    """Clear pipeline outputs and checkpoints; preserve source data."""
    for path in [
        CHECKPOINT_FILE, ACCEPTED_FILE, REJECTED_FILE,
        AGGREGATE_FILE, RUN_LOG_FILE
    ]:
        if path.exists():
            path.unlink()


# -------------------- 8. AUTOMATED TESTS --------------------

def run_tests():
    print("\n" + "=" * 60)
    print("RUNNING AUTOMATED TESTS")
    print("=" * 60)

    passed = 0
    failed = []

    def check(name, condition):
        nonlocal passed
        if condition:
            passed += 1
            print(f"PASS: {name}")
        else:
            failed.append(name)
            print(f"FAIL: {name}")

    # Test record validation.
    valid_row = {
        "event_id": "EVT_TEST",
        "order_id": "ORD_TEST",
        "customer_id": "CUST00001",
        "event_time": "2026-10-01T10:00:00Z",
        "amount": 100.0,
        "status": "paid",
        "source_offset": 1
    }

    check("Valid record", validate_record(valid_row) == [])

    missing_row = {**valid_row, "customer_id": None}
    check(
        "Missing required field",
        "missing_customer_id" in validate_record(missing_row)
    )

    bad_amount = {**valid_row, "amount": -1}
    check("Negative amount", "negative_amount" in validate_record(bad_amount))

    bad_status = {**valid_row, "status": "unknown"}
    check("Invalid status", "invalid_status" in validate_record(bad_status))

    bad_time = {**valid_row, "event_time": "not-a-date"}
    check("Malformed timestamp", "invalid_event_time" in validate_record(bad_time))

    bad_customer = {**valid_row, "customer_id": "UNKNOWN"}
    check(
        "Invalid customer reference",
        "invalid_customer_reference" in validate_record(bad_customer)
    )

    try:
        validate_schema(pd.DataFrame(columns=REQUIRED_COLUMNS + ["extra"]))
        schema_ok = False
    except ValueError:
        schema_ok = True

    check("Unexpected schema change", schema_ok)

    # Integration tests run in a temporary workspace.
    main_dir = PROJECT_DIR
    test_dir = Path(tempfile.mkdtemp(prefix="stream_pipeline_test_"))

    try:
        configure_project(test_dir)

        # Empty source test
        pd.DataFrame(columns=REQUIRED_COLUMNS).to_csv(
            SOURCE_FILE, index=False
        )
        empty_result = process_pipeline()
        check(
            "Empty input",
            empty_result["status"] == "success"
            and empty_result["input_count"] == 0
        )

        # Integration and retry tests
        reset_outputs()
        generate_events(30, seed=42).to_csv(SOURCE_FILE, index=False)

        failed_run = process_pipeline(
            batch_size=10, fail_after_write=True
        )
        retry_run = process_pipeline(batch_size=10)

        accepted = read_existing(ACCEPTED_FILE)
        rejected = read_existing(REJECTED_FILE)
        aggregates = read_existing(AGGREGATE_FILE)

        check("Simulated failure recorded", failed_run["status"] == "failed")
        check("Retry succeeds", retry_run["status"] == "success")
        check(
            "No duplicate accepted event IDs",
            accepted["event_id"].is_unique
        )
        check(
            "No duplicate accepted source offsets",
            accepted["source_offset"].is_unique
        )
        check(
            "No duplicate rejected source offsets",
            rejected["source_offset"].is_unique
        )
        check(
            "Source reconciliation",
            len(accepted) + len(rejected) == 31
        )
        check(
            "Duplicate delivery quarantined",
            rejected["reject_reason"].astype(str)
            .str.contains("duplicate_event_id").any()
        )
        check(
            "Late event detected",
            accepted["is_late"].astype(str).str.lower().eq("true").any()
        )
        check(
            "Aggregate event counts reconcile",
            int(aggregates["event_count"].sum()) == len(accepted)
        )

        # A further rerun must not change output row counts.
        accepted_before = len(accepted)
        rejected_before = len(rejected)

        rerun = process_pipeline(batch_size=10)
        accepted_after = len(read_existing(ACCEPTED_FILE))
        rejected_after = len(read_existing(REJECTED_FILE))

        check(
            "Same-source rerun is idempotent",
            rerun["status"] == "success"
            and accepted_before == accepted_after
            and rejected_before == rejected_after
        )

    finally:
        configure_project(main_dir)
        shutil.rmtree(test_dir, ignore_errors=True)

    print(f"\nTests passed: {passed}")
    print(f"Tests failed: {len(failed)}")

    if failed:
        raise AssertionError(f"Failed tests: {failed}")

    print("ALL AUTOMATED TESTS PASSED")


# -------------------- 9. DOCUMENTATION --------------------

def create_documentation():
    readme = """
# Streaming Event Processing Pipeline

## Objective
Simulate a continuously arriving stream of order events using micro-batches.
Validate records, deduplicate event IDs, detect late events, maintain
checkpoints, quarantine bad data, and publish hourly aggregates.

## Architecture
Synthetic producer -> CSV source -> micro-batch ingestion -> validation
-> accepted events / quarantine -> hourly aggregates and operational logs.

## Technology
Python, Pandas, NumPy, CSV, JSON, Google Colab.

## Source schema
event_id: required stable event identifier
order_id: required business order identifier
customer_id: required customer reference
event_time: ISO-8601 UTC timestamp
amount: finite non-negative number
status: created, paid, or cancelled
source_offset: positive unique source position

## Output files
output/accepted_events.csv
output/quarantine/rejected_events.csv
output/aggregates.csv
state/checkpoint.json
logs/run_metrics.csv

## Recovery
The checkpoint records the last committed source offset and maximum observed
event time. Output files are written before checkpoint advancement. A retry
skips already persisted source offsets. Event IDs prevent duplicate business
events from being accepted.

## Late events
Events more than 10 minutes behind the maximum event time observed are
flagged with is_late=True. They are retained rather than discarded.

## Data quality
Invalid records are quarantined with their original source fields and a
reject_reason. Unexpected schema changes fail the run.

## Reproducibility
The synthetic data generator uses a fixed random seed. The sample contains
intentional invalid records and duplicate delivery.

## Limitations
This is a single-process streaming simulation, not a production message
broker. CSV files and JSON checkpoints do not provide a fully transactional
commit across outputs and state. Stronger production guarantees require
transactional storage, durable state, and concurrency control.

## Run
Open the notebook in Google Colab and run the complete code cell.
"""

    dictionary = """
# Data Dictionary

| Field | Type | Meaning |
|---|---|---|
| event_id | string | Stable event identifier |
| order_id | string | Order identifier |
| customer_id | string | Customer reference |
| event_time | UTC timestamp | Business event time |
| amount | float | Non-negative amount |
| status | string | Business event status |
| source_offset | integer | Source record position |
| run_id | string | Pipeline run identifier |
| is_late | boolean | Late event flag |
| reject_reason | string | Quarantine reason |
"""

    design = """
# Design and Trade-offs

- CSV and JSON keep outputs inspectable in Colab.
- Micro-batches provide manageable processing units.
- Source offsets support restart and replay protection.
- Event IDs provide business-level deduplication.
- Quarantine preserves invalid records for investigation.
- Late events are flagged and retained.
- Aggregates are rebuilt from accepted event history for reconciliation.
- This implementation is intended for learning and demonstration.
- A database, transactional sink, or Spark Structured Streaming is a
  reasonable next step for stronger recovery and larger workloads.
"""

    (PROJECT_DIR / "README.md").write_text(readme.strip() + "\n")
    (PROJECT_DIR / "DATA_DICTIONARY.md").write_text(dictionary.strip() + "\n")
    (PROJECT_DIR / "DESIGN_NOTES.md").write_text(design.strip() + "\n")


# -------------------- 10. MAIN EXECUTION --------------------

def main():
    print("=" * 60)
    print("STREAMING EVENT PROCESSING PIPELINE")
    print("=" * 60)

    # Generate the small reproducible dataset.
    reset_outputs()
    sample = generate_events(n=1000, seed=42)
    sample.to_csv(SOURCE_FILE, index=False)

    print("\nSample dataset created:", SOURCE_FILE)
    print("Source rows:", len(sample))

    # Process sample in micro-batches.
    print("\nPROCESSING SAMPLE DATA")
    sample_result = process_pipeline(batch_size=500)
    assert sample_result["status"] == "success"

    print("\nSAMPLE AGGREGATES")
    display(read_existing(AGGREGATE_FILE).head(10))

    print("\nSAMPLE QUARANTINE")
    display(read_existing(REJECTED_FILE).head(10))

    # Show reconciliation.
    accepted = read_existing(ACCEPTED_FILE)
    rejected = read_existing(REJECTED_FILE)
    aggregates = read_existing(AGGREGATE_FILE)

    print("\nSAMPLE RECONCILIATION")
    print("Source rows:", len(sample))
    print("Accepted rows:", len(accepted))
    print("Rejected rows:", len(rejected))
    print("Accepted + rejected:", len(accepted) + len(rejected))
    print("Aggregate event count:", int(aggregates["event_count"].sum()))
    print("Accepted amount:", round(accepted["amount"].sum(), 2))
    print("Aggregate amount:", round(aggregates["total_amount"].sum(), 2))

    assert len(accepted) + len(rejected) == len(sample)
    assert int(aggregates["event_count"].sum()) == len(accepted)
    assert round(accepted["amount"].sum(), 2) == round(
        aggregates["total_amount"].sum(), 2
    )

    # Automated tests use an isolated temporary workspace.
    run_tests()

    # Clean benchmark run.
    print("\n" + "=" * 60)
    print("100,000-RECORD PERFORMANCE BENCHMARK")
    print("=" * 60)

    reset_outputs()
    large = generate_events(n=100_000, seed=42)
    large.to_csv(SOURCE_FILE, index=False)
    print("Generated source rows:", len(large))

    start = time.perf_counter()
    benchmark_result = process_pipeline(batch_size=10_000)
    elapsed = time.perf_counter() - start

    assert benchmark_result["status"] == "success"

    final_accepted = read_existing(ACCEPTED_FILE)
    final_rejected = read_existing(REJECTED_FILE)
    final_aggregates = read_existing(AGGREGATE_FILE)

    throughput = benchmark_result["input_count"] / max(elapsed, 0.001)

    print(f"\nElapsed seconds: {elapsed:.2f}")
    print(f"Rows processed: {benchmark_result['input_count']}")
    print(f"Throughput: {throughput:.0f} rows/second")
    print("Accepted:", len(final_accepted))
    print("Rejected:", len(final_rejected))

    # Final correctness assertions.
    assert len(final_accepted) + len(final_rejected) == len(large)
    assert final_accepted["event_id"].is_unique
    assert final_accepted["source_offset"].is_unique
    assert int(final_aggregates["event_count"].sum()) == len(final_accepted)

    assert round(final_accepted["amount"].sum(), 2) == round(
        final_aggregates["total_amount"].sum(), 2
    )

    # Write project documentation.
    create_documentation()

    print("\n" + "=" * 60)
    print("PROJECT COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print("Project folder:", PROJECT_DIR)
    print("\nGenerated deliverables:")

    for path in sorted(PROJECT_DIR.rglob("*")):
        if path.is_file():
            print(" -", path.relative_to(PROJECT_DIR))

    print("\nAll final reconciliation assertions passed.")
    print("Use the Files panel in Colab to inspect your output files.")


main()


STREAMING EVENT PROCESSING PIPELINE

Sample dataset created: /content/streaming_event_pipeline/data/events.csv
Source rows: 1001

PROCESSING SAMPLE DATA
{
  "run_id": "c4c514f1-9642-44cb-8ce3-1f74495f02c2",
  "started_at": "2026-10-29T11:14:55.464271+00:00",
  "ended_at": "2026-10-29T11:14:57.746847+00:00",
  "status": "success",
  "input_count": 1001,
  "accepted_count": 996,
  "rejected_count": 5,
  "duplicate_count": 1,
  "last_source_offset": 1001,
  "elapsed_seconds": 2.2825,
  "error_message": null
}

SAMPLE AGGREGATES


,event_hour,status,event_count,total_amount
0,2026-09-01 00:00:00+00:00,paid,1,4750.80
1,2026-10-01 00:00:00+00:00,cancelled,2,2698.51
2,2026-10-01 00:00:00+00:00,created,2,1386.32
3,2026-10-01 00:00:00+00:00,paid,7,20622.79
4,2026-10-01 01:00:00+00:00,cancelled,2,7603.22
5,2026-10-01 01:00:00+00:00,created,1,1066.74
6,2026-10-01 01:00:00+00:00,paid,4,5556.88
7,2026-10-01 02:00:00+00:00,cancelled,2,4558.64
8,2026-10-01 02:00:00+00:00,created,1,2046.70
9,2026-10-01 02:00:00+00:00,paid,2,5936.66



SAMPLE QUARANTINE


,event_id,order_id,customer_id,event_time,amount,status,source_offset,run_id,reject_reason
0,EVT00000006,ORD00000006,NaN,2026-10-07T10:44:15+00:00,1970.71,cancelled,6,c4c514f1-9642-44cb-8ce3-1f74495f02c2,missing_customer_id
1,EVT00000008,ORD00000008,CUST99999,2026-10-04T12:36:37+00:00,1602.59,created,8,c4c514f1-9642-44cb-8ce3-1f74495f02c2,invalid_customer_reference
2,EVT00000010,ORD00000010,CUST00095,2026-10-04T15:24:22+00:00,-25.00,paid,10,c4c514f1-9642-44cb-8ce3-1f74495f02c2,negative_amount
3,EVT00000012,ORD00000012,CUST00976,2026-10-06T14:22:41+00:00,3002.48,unknown,12,c4c514f1-9642-44cb-8ce3-1f74495f02c2,invalid_status
4,EVT00000003,ORD00000003,CUST00655,2026-10-04T21:47:50+00:00,653.86,cancelled,1001,c4c514f1-9642-44cb-8ce3-1f74495f02c2,duplicate_event_id



SAMPLE RECONCILIATION
Source rows: 1001
Accepted rows: 996
Rejected rows: 5
Accepted + rejected: 1001
Aggregate event count: 996
Accepted amount: 2538506.14
Aggregate amount: 2538506.14

RUNNING AUTOMATED TESTS
PASS: Valid record
PASS: Missing required field
PASS: Negative amount
PASS: Invalid status
PASS: Malformed timestamp
PASS: Invalid customer reference
PASS: Unexpected schema change
{
  "run_id": "f4145e4b-9d6e-443c-9a8c-c87cc56c7d4c",
  "started_at": "2026-10-29T11:14:57.954417+00:00",
  "ended_at": "2026-10-29T11:14:57.961296+00:00",
  "status": "success",
  "input_count": 0,
  "accepted_count": 0,
  "rejected_count": 0,
  "duplicate_count": 0,
  "last_source_offset": 0,
  "elapsed_seconds": 0.0068,
  "error_message": null
}
PASS: Empty input


2026-10-29 11:14:58,078 | ERROR | Pipeline run failed: b533aa35-90bc-4226-8a0d-2b2c6170f7f2
Traceback (most recent call last):
  File "/tmp/ipykernel_4469/904799728.py", line 420, in process_pipeline
    raise RuntimeError("Simulated failure before checkpoint")
RuntimeError: Simulated failure before checkpoint


{
  "run_id": "b533aa35-90bc-4226-8a0d-2b2c6170f7f2",
  "started_at": "2026-10-29T11:14:57.993518+00:00",
  "ended_at": "2026-10-29T11:14:58.082550+00:00",
  "status": "failed",
  "input_count": 10,
  "accepted_count": 7,
  "rejected_count": 3,
  "duplicate_count": 0,
  "last_source_offset": 0,
  "elapsed_seconds": 0.089,
  "error_message": "Simulated failure before checkpoint"
}
{
  "run_id": "cad4e663-ec03-4391-911a-1d05cbe6c1f6",
  "started_at": "2026-10-29T11:14:58.091889+00:00",
  "ended_at": "2026-10-29T11:14:58.508356+00:00",
  "status": "success",
  "input_count": 31,
  "accepted_count": 19,
  "rejected_count": 2,
  "duplicate_count": 1,
  "last_source_offset": 31,
  "elapsed_seconds": 0.4164,
  "error_message": null
}
PASS: Simulated failure recorded
PASS: Retry succeeds
PASS: No duplicate accepted event IDs
PASS: No duplicate accepted source offsets
PASS: No duplicate rejected source offsets
PASS: Source reconciliation
PASS: Duplicate delivery quarantined
PASS: Late event det

In [ ]:

import os

project_dir = "/content/streaming_event_pipeline"

print("Does the project folder exist?", os.path.exists(project_dir))

if os.path.exists(project_dir):
    for root, dirs, files in os.walk(project_dir):
        print("\nFolder:", root)
        for file in files:
            print("  -", file)
else:
    print("Project folder not found.")


Does the project folder exist? True

Folder: /content/streaming_event_pipeline
  - DESIGN_NOTES.md
  - README.md
  - DATA_DICTIONARY.md

Folder: /content/streaming_event_pipeline/logs
  - run_metrics.csv

Folder: /content/streaming_event_pipeline/output
  - aggregates.csv
  - accepted_events.csv

Folder: /content/streaming_event_pipeline/output/quarantine
  - rejected_events.csv

Folder: /content/streaming_event_pipeline/data
  - events.csv

Folder: /content/streaming_event_pipeline/state
  - checkpoint.json


Folder: /content/streaming_event_pipeline
  - README.md
  - DATA_DICTIONARY.md
  - DESIGN_NOTES.md

Folder: /content/streaming_event_pipeline/output
  - accepted_events.csv
  - aggregates.csv

Folder: /content/streaming_event_pipeline/output/quarantine
  - rejected_events.csv

Folder: /content/streaming_event_pipeline/state
  - checkpoint.json

Folder: /content/streaming_event_pipeline/logs
  - run_metrics.csv

In [ ]:

# Check whether the pipeline's test function exists
print("run_tests exists:", "run_tests" in globals())

# Run the tests if the function exists
if "run_tests" in globals():
    run_tests()
else:
    print("run_tests() was not found. Check your earlier code cells.")


2026-10-29 11:29:12,758 | ERROR | Pipeline run failed: 1c9fcb96-5bbf-4679-bda3-3466962fe7b1
Traceback (most recent call last):
  File "/tmp/ipykernel_4469/904799728.py", line 420, in process_pipeline
    raise RuntimeError("Simulated failure before checkpoint")
RuntimeError: Simulated failure before checkpoint


run_tests exists: True

RUNNING AUTOMATED TESTS
PASS: Valid record
PASS: Missing required field
PASS: Negative amount
PASS: Invalid status
PASS: Malformed timestamp
PASS: Invalid customer reference
PASS: Unexpected schema change
{
  "run_id": "d6119652-9acc-4189-8a4d-8d232cdffbe4",
  "started_at": "2026-10-29T11:29:12.720282+00:00",
  "ended_at": "2026-10-29T11:29:12.726001+00:00",
  "status": "success",
  "input_count": 0,
  "accepted_count": 0,
  "rejected_count": 0,
  "duplicate_count": 0,
  "last_source_offset": 0,
  "elapsed_seconds": 0.0057,
  "error_message": null
}
PASS: Empty input
{
  "run_id": "1c9fcb96-5bbf-4679-bda3-3466962fe7b1",
  "started_at": "2026-10-29T11:29:12.733514+00:00",
  "ended_at": "2026-10-29T11:29:12.759591+00:00",
  "status": "failed",
  "input_count": 10,
  "accepted_count": 7,
  "rejected_count": 3,
  "duplicate_count": 0,
  "last_source_offset": 0,
  "elapsed_seconds": 0.0261,
  "error_message": "Simulated failure before checkpoint"
}
{
  "run_id": "6cd

In [ ]:

import os

base = "/content/streaming_event_pipeline"

required_files = [
    "data/events.csv",
    "output/accepted_events.csv",
    "output/quarantine/rejected_events.csv",
    "output/aggregates.csv",
    "state/checkpoint.json",
    "logs/run_metrics.csv",
]

for file in required_files:
    path = os.path.join(base, file)
    print(("PASS" if os.path.isfile(path) else "FAIL"), "-", file)


PASS - data/events.csv
PASS - output/accepted_events.csv
PASS - output/quarantine/rejected_events.csv
PASS - output/aggregates.csv
PASS - state/checkpoint.json
PASS - logs/run_metrics.csv


In [ ]:

import os
import pandas as pd

base = "/content/streaming_event_pipeline"

files = {
    "Source events": "data/events.csv",
    "Accepted events": "output/accepted_events.csv",
    "Rejected events": "output/quarantine/rejected_events.csv",
    "Aggregates": "output/aggregates.csv",
    "Run metrics": "logs/run_metrics.csv",
}

for name, relative_path in files.items():
    path = os.path.join(base, relative_path)
    print(f"\n{name}:")

    if os.path.isfile(path):
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        display(df.head(3))
    else:
        print("File not found")



Source events:
Rows: 100001
Columns: ['event_id', 'order_id', 'customer_id', 'event_time', 'amount', 'status', 'source_offset']


,event_id,order_id,customer_id,event_time,amount,status,source_offset
0,EVT00000001,ORD00000001,CUST00090,2026-10-06T03:31:29+00:00,3699.22,paid,1
1,EVT00000002,ORD00000002,CUST00774,2026-10-07T04:12:32+00:00,3311.25,paid,2
2,EVT00000003,ORD00000003,CUST00655,2026-10-05T00:03:25+00:00,4377.87,paid,3



Accepted events:
Rows: 99996
Columns: ['event_id', 'order_id', 'customer_id', 'event_time', 'amount', 'status', 'source_offset', 'run_id', 'is_late']


,event_id,order_id,customer_id,event_time,amount,status,source_offset,run_id,is_late
0,EVT00000001,ORD00000001,CUST00090,2026-10-06T03:31:29+00:00,3699.22,paid,1,d6d0f17d-cce0-4771-9579-be8603ba6df3,False
1,EVT00000002,ORD00000002,CUST00774,2026-10-07T04:12:32+00:00,3311.25,paid,2,d6d0f17d-cce0-4771-9579-be8603ba6df3,False
2,EVT00000003,ORD00000003,CUST00655,2026-10-05T00:03:25+00:00,4377.87,paid,3,d6d0f17d-cce0-4771-9579-be8603ba6df3,True



Rejected events:
Rows: 5
Columns: ['event_id', 'order_id', 'customer_id', 'event_time', 'amount', 'status', 'source_offset', 'run_id', 'reject_reason']


,event_id,order_id,customer_id,event_time,amount,status,source_offset,run_id,reject_reason
0,EVT00000006,ORD00000006,NaN,2026-10-05T06:55:40+00:00,2924.15,paid,6,d6d0f17d-cce0-4771-9579-be8603ba6df3,missing_customer_id
1,EVT00000008,ORD00000008,CUST99999,2026-10-01T23:39:16+00:00,4647.04,created,8,d6d0f17d-cce0-4771-9579-be8603ba6df3,invalid_customer_reference
2,EVT00000010,ORD00000010,CUST00095,2026-10-04T16:32:21+00:00,-25.00,paid,10,d6d0f17d-cce0-4771-9579-be8603ba6df3,negative_amount



Aggregates:
Rows: 505
Columns: ['event_hour', 'status', 'event_count', 'total_amount']


,event_hour,status,event_count,total_amount
0,2026-09-01 00:00:00+00:00,paid,1,4561.23
1,2026-10-01 00:00:00+00:00,cancelled,82,208392.48
2,2026-10-01 00:00:00+00:00,created,150,408419.41



Run metrics:
Rows: 1
Columns: ['run_id', 'started_at', 'ended_at', 'status', 'input_count', 'accepted_count', 'rejected_count', 'duplicate_count', 'last_source_offset', 'elapsed_seconds', 'error_message']


,run_id,started_at,ended_at,status,input_count,accepted_count,rejected_count,duplicate_count,last_source_offset,elapsed_seconds,error_message
0,d6d0f17d-cce0-4771-9579-be8603ba6df3,2026-10-29T11:15:03.841970+00:00,2026-10-29T11:18:03.880769+00:00,success,100001,99996,5,1,100001,180.0387,NaN


In [ ]:

import os
import pandas as pd
import numpy as np

base = "/content/streaming_event_pipeline"

accepted = pd.read_csv(
    f"{base}/output/accepted_events.csv"
)
aggregates = pd.read_csv(
    f"{base}/output/aggregates.csv"
)

# Calculate the correct hourly aggregates from accepted events
accepted["event_time"] = pd.to_datetime(
    accepted["event_time"], utc=True
)
accepted["hour"] = accepted["event_time"].dt.floor("h")

expected = (
    accepted.groupby(["hour", "status"], as_index=False)
    .agg(
        expected_count=("event_id", "size"),
        expected_amount=("amount", "sum")
    )
)

aggregates["event_hour"] = pd.to_datetime(
    aggregates["event_hour"], utc=True
)

actual = aggregates.rename(columns={
    "event_count": "actual_count",
    "total_amount": "actual_amount"
})

check = expected.merge(
    actual,
    left_on=["hour", "status"],
    right_on=["event_hour", "status"],
    how="outer",
    indicator=True
)

matched = check[check["_merge"] == "both"]

counts_match = (
    len(expected) == len(actual)
    and (check["_merge"] == "both").all()
    and (matched["expected_count"] == matched["actual_count"]).all()
)

amounts_match = np.allclose(
    matched["expected_amount"],
    matched["actual_amount"],
    rtol=1e-6,
    atol=0.01
)

print("Expected hourly groups:", len(expected))
print("Actual aggregate groups:", len(actual))
print("Counts match:", counts_match)
print("Amounts match:", amounts_match)

if counts_match and amounts_match:
    print("PASS: Aggregation is correct.")
else:
    print("FAIL: Investigate aggregation logic.")
    display(check[check["_merge"] != "both"].head(10))
    display(matched.head(10))


Expected hourly groups: 505
Actual aggregate groups: 505
Counts match: True
Amounts match: True
PASS: Aggregation is correct.


In [ ]:

if "run_tests" in globals():
    run_tests()
else:
    print("run_tests() not found. Run the original pipeline cell first.")


2026-10-29 11:34:36,782 | ERROR | Pipeline run failed: af74979a-9e88-4a55-bdca-63797d130129
Traceback (most recent call last):
  File "/tmp/ipykernel_4469/904799728.py", line 420, in process_pipeline
    raise RuntimeError("Simulated failure before checkpoint")
RuntimeError: Simulated failure before checkpoint



RUNNING AUTOMATED TESTS
PASS: Valid record
PASS: Missing required field
PASS: Negative amount
PASS: Invalid status
PASS: Malformed timestamp
PASS: Invalid customer reference
PASS: Unexpected schema change
{
  "run_id": "a61c8ffc-05df-407d-ab60-8061b9980272",
  "started_at": "2026-10-29T11:34:36.743243+00:00",
  "ended_at": "2026-10-29T11:34:36.748403+00:00",
  "status": "success",
  "input_count": 0,
  "accepted_count": 0,
  "rejected_count": 0,
  "duplicate_count": 0,
  "last_source_offset": 0,
  "elapsed_seconds": 0.0051,
  "error_message": null
}
PASS: Empty input
{
  "run_id": "af74979a-9e88-4a55-bdca-63797d130129",
  "started_at": "2026-10-29T11:34:36.756086+00:00",
  "ended_at": "2026-10-29T11:34:36.783538+00:00",
  "status": "failed",
  "input_count": 10,
  "accepted_count": 7,
  "rejected_count": 3,
  "duplicate_count": 0,
  "last_source_offset": 0,
  "elapsed_seconds": 0.0274,
  "error_message": "Simulated failure before checkpoint"
}
{
  "run_id": "6151dce4-96ae-498c-852e-e3

In [8]:

import os
import shutil
from google.colab import files

project_dir = "/content/streaming_event_pipeline"

if os.path.isdir(project_dir):
    zip_path = shutil.make_archive(
        "/content/streaming_event_pipeline",
        "zip",
        root_dir="/content",
        base_dir="streaming_event_pipeline"
    )

    print("Project ZIP created successfully!")
    files.download(zip_path)
else:
    print("Project folder not found. Please check the folder path.")


Project ZIP created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:

import os

project_dir = "/content/streaming_event_pipeline"

print("Does project folder exist?", os.path.isdir(project_dir))

if os.path.isdir(project_dir):
    print("\nFiles inside project folder:")
    for root, dirs, files in os.walk(project_dir):
        for name in files:
            print(os.path.join(root, name))
else:
    print("\nProject folder was not found.")
    print("\nFolders currently available in /content:")
    print(os.listdir("/content"))


Does project folder exist? True

Files inside project folder:
/content/streaming_event_pipeline/DESIGN_NOTES.md
/content/streaming_event_pipeline/README.md
/content/streaming_event_pipeline/DATA_DICTIONARY.md
/content/streaming_event_pipeline/logs/run_metrics.csv
/content/streaming_event_pipeline/output/aggregates.csv
/content/streaming_event_pipeline/output/accepted_events.csv
/content/streaming_event_pipeline/output/quarantine/rejected_events.csv
/content/streaming_event_pipeline/data/events.csv
/content/streaming_event_pipeline/state/checkpoint.json


In [10]:

import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "/content/streaming_event_pipeline",
    "zip",
    root_dir="/content",
    base_dir="streaming_event_pipeline"
)

print("ZIP file created at:", zip_path)
print("File size:", round(__import__("os").path.getsize(zip_path) / (1024 * 1024), 2), "MB")

files.download(zip_path)


ZIP file created at: /content/streaming_event_pipeline.zip
File size: 3.98 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
